In [2]:
import heapq

def dijkstra(graph, start):
    dist = {node: float('inf') for node in graph}
    dist[start] = 0
    pq = [(0, start)]


    while pq:
        d,u = heapq.heappop(pq)
        if d > dist[u]:
            continue
        for v, peso in graph[u]:
            if dist[u] + peso < dist[v]:
                dist[v] = dist[u] + peso
                heapq.heappush(pq, (dist[v], v))

    return dist


graph = {
    'A': [('B', 1), ('C', 4)],
    'B': [('A', 1), ('C', 2), ('D', 5)],
    'C': [('A', 4), ('B', 2)],
    'D': [('B', 5), ('E', 3)],
    'E': [('D', 3)]

}


print(dijkstra(graph, 'A'))

{'A': 0, 'B': 1, 'C': 3, 'D': 6, 'E': 9}


In [4]:
def bellman_ford(vertices, edges, start):
    dist = {v: float('inf') for v in vertices}
    dist[start] = 0

    for _ in range(len(vertices) - 1):
        for u,v,w in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w

    for u,v,w in edges:
        if dist[u] + w < dist[v]:
            raise ValueError("ciclo negativo")

    return dist


vertices = ['A','B','C','D']

edges = [
    ('A','B',1),
    ('A','C',4),
    ('B','C',2),
    ('B','D',5),
    ('C','D',3)
]

print(bellman_ford(vertices, edges, 'A'))

{'A': 0, 'B': 1, 'C': 3, 'D': 6}


In [5]:
def floyd_warsall(n, edges):
    INF = float('inf')
    dist = [[INF] * n for _ in range(n)]
    for i in range(n):
        dist[i][i] = 0
    for u,v,w in edges:
        dist[u][v] = w


    for k in range(n):
        for i in range(n):
            for j in range(n):
                if dist[i][k] + dist[k][j] < dist[i][j]:
                  dist[i][j] = dist[i][k] + dist[k][j]

    return dist


edges = [(0,1,3),(0,2,6),(1,2,1)]
dist = floyd_warsall(3, edges)
print(dist)


[[0, 3, 4], [inf, 0, 1], [inf, inf, 0]]


In [8]:
from collections import deque

def bfs_shortest(graph, start, end):
    queue = deque([(start, [start])])
    visited = {start}

    while queue:
        node, path = queue.popleft()
        if node == end:
            return len(path) - 1, path

        for neighbor in graph[node]:
            if neighbor not in visited:
               visited.add(neighbor)
               queue.append((neighbor, path + [neighbor]))

    return -1, []


graph = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'F'],
    'D': ['B'],
    'E': ['B', 'F']
}

dist, path = bfs_shortest(graph, 'A','D')
print(dist, path)


2 ['A', 'B', 'D']


In [9]:
def floyd_warshall(graph):
    nodes = list(graph.keys())
    n = len(nodes)
    idx = {v: i for i, v in enumerate(nodes)}

    INF = float('inf')
    dist = [[INF] * n for _ in range(n)]
    nxt = [[None] * n for _ in range(n)]

    for u, neighbors in graph.items():
        i = idx[u]
        dist[i][i] = 0
        for v, w in neighbors.items():
            j = idx[v]
            dist[i][j] = w
            nxt[i][j] = j

    for k in range(n):
        for i in range(n):
            for j in range(n):
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]
                    nxt[i][j] = nxt[i][k]

    for i in range(n):
        if dist[i][i] < 0:
            raise ValueError("ciclo negativo")

    return dist, nxt, nodes


def get_path(nxt, nodes, src, dst):
    idx = {v: i for i, v in enumerate(nodes)}
    i,j = idx[src], idx[dst]
    if nxt[i][j] is None:
        return []
    path = [src]
    while i != j:
        i = nxt[i][j]
        path.append(nodes[i])
    return path


graph = {
    'A': {'B': 3, 'C': 6},
    'B': {'A': 3, 'D': 1, 'E': 2},
    'C': {'A': 6, 'F': 4},
    'D': {'B': 1},
    'E': {'B': 2, 'F': 1},
    'F': {'C': 4, 'E': 1}
}

dist, nxt, nodes = floyd_warshall(graph)
print(get_path(nxt, nodes, 'A','E'))


['A', 'B', 'E']


In [12]:
import heapq

def bellman_ford(nodes, edges, src):
    dist = {v: float('inf') for v in nodes}
    dist[src] = 0
    for _ in range(len(nodes) - 1):
        for u, v, w in edges:
            # Ensure u and v are in dist before accessing
            if u in dist and v in dist and dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
    return dist


def dijkstra(graph, start, h):
    dist = {v: float('inf') for v in graph}
    dist[start] = 0
    pq = [(0,start)]
    while pq:
        d,u = heapq.heappop(pq)
        if d > dist[u]: continue
        for v, w in graph[u]:
            w_new = w + h[u] - h[v]
            if dist[u] + w_new < dist[v]:
                dist[v] = dist[u] + w_new
                heapq.heappush(pq, (dist[v], v))

    return {v: dist[v] - h[start] + h[v] for v in dist}


def johnson(graph):
    # Collect all unique nodes from the graph (keys and values)
    nodes_set = set(graph.keys())
    for u in graph:
        for v in graph[u]:
            nodes_set.add(v)
    nodes = list(nodes_set)

    edges = [(u,v,w) for u in graph for v, w in graph[u].items()]
    q = '__q__'
    nodes_q = nodes + [q]
    edges_q = edges + [(q,v,0) for v in nodes]

    h = bellman_ford(nodes_q, edges_q,q)

    graph_rw = {u: [(v,w + h[u] - h[v]) for v,w in nbrs.items()]
                        for u, nbrs in graph.items()}

    return {u: dijkstra(graph_rw, u, h) for u in nodes}


graph = {
    'A': {'B': 3, 'C': 6},
    'B': {'A': 3, 'D': 1, 'E': 2},
    'C': {'A': 6, 'F': 4},
    'D': {'B': 1},
    'E': {'B': 2, 'F': 1},
    'F': {'C': 4, 'E': 1} # Added 'F' as a key to ensure all nodes are in graph.keys()
}



all_dist = johnson(graph)
print(all_dist['A'])

{'A': 0, 'B': 3, 'C': 6, 'D': 4, 'E': 5, 'F': 6}


In [14]:
from collections import deque

def bfs_from(graph, start):
    dist = {start: 0}
    queue = deque([start])
    while queue:
        u = queue.popleft()
        # Check if u exists as a key in the graph before trying to access its neighbors
        if u in graph:
            for v in graph[u]:
                if v not in dist:
                    dist[v] = dist[u] + 1
                    queue.append(v)
    return dist


def all_pairs_bfs(graph):
    return {u: bfs_from(graph, u) for u in graph}



graph = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'F'],
    'D': ['B'],
    'E': ['B', 'F'],
    'F': [] # Added 'F' as a key to prevent KeyError
}

apsp = all_pairs_bfs(graph)
print(apsp['A'])


diameter = max(
    d for dists in apsp.values()
    for d in dists.values()
)

print(f"diametro: {diameter}")


{'A': 0, 'B': 1, 'C': 1, 'D': 2, 'E': 2, 'F': 2}
diametro: 3
